<a href="https://colab.research.google.com/github/SaravananSDATAENGINEER/Agentic_AI/blob/main/rag_langchain_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



In [ ]:
# 1) Load environment variables from .env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Put it in a .env file.")

In [ ]:
# 2) Load the policy document
loader = TextLoader("bank_policy.txt", encoding="utf-8")

In [ ]:
documents = loader.load()

In [ ]:
documents

[Document(metadata={'source': 'bank_policy.txt'}, page_content='BANK POLICY MANUAL (Sample) — Retail Banking\nVersion: 1.0\nEffective Date: 01-Jan-2026\n\n1. Savings Account — Minimum Balance & Fees\n1.1 Minimum monthly average balance: SAR 3,000.\n1.2 If monthly average balance falls below SAR 3,000, a maintenance fee of SAR 25 is charged at month-end.\n1.3 Student savings accounts are exempt from minimum balance rules and maintenance fees.\n1.4 Accounts opened via digital channels may receive a 3-month fee waiver from the opening date.\n\n2. Current Account — Cheque Book & Charges\n2.1 Cheque book issuance is available to customers with a current account active for at least 30 days.\n2.2 Cheque book issuance fee: SAR 15 per cheque book.\n2.3 If a cheque is returned due to insufficient funds, a penalty of SAR 150 is applied per incident.\n\n3. Personal Loan — Eligibility & Pricing\n3.1 Minimum age: 21 years. Maximum age at loan maturity: 60 years.\n3.2 Minimum monthly salary: SAR 5,00

In [ ]:
# 3) Split into chunks for better retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(documents)

In [ ]:
chunks

[Document(metadata={'source': 'bank_policy.txt'}, page_content='BANK POLICY MANUAL (Sample) — Retail Banking\nVersion: 1.0\nEffective Date: 01-Jan-2026\n\n1. Savings Account — Minimum Balance & Fees\n1.1 Minimum monthly average balance: SAR 3,000.\n1.2 If monthly average balance falls below SAR 3,000, a maintenance fee of SAR 25 is charged at month-end.\n1.3 Student savings accounts are exempt from minimum balance rules and maintenance fees.\n1.4 Accounts opened via digital channels may receive a 3-month fee waiver from the opening date.'),
 Document(metadata={'source': 'bank_policy.txt'}, page_content='2. Current Account — Cheque Book & Charges\n2.1 Cheque book issuance is available to customers with a current account active for at least 30 days.\n2.2 Cheque book issuance fee: SAR 15 per cheque book.\n2.3 If a cheque is returned due to insufficient funds, a penalty of SAR 150 is applied per incident.'),
 Document(metadata={'source': 'bank_policy.txt'}, page_content='3. Personal Loan —

In [ ]:
# 4) Create embeddings + vector store (FAISS in-memory)
embeddings = OpenAIEmbeddings(api_key=api_key)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=api_key)

In [ ]:
# 6) Prompt (inject retrieved context)
prompt = ChatPromptTemplate.from_template(
    """You are a banking policy assistant.
Answer ONLY using the context below. If the answer is not in the context, say: "I don't know based on the provided policy."

Context:
{context}

Question:
{question}

Answer:"""
)

In [ ]:
# Helper to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# 7) RAG chain (modern LCEL style)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# 8) Sample questions to test
questions = [
    "What is the minimum monthly average balance for a savings account?",
    "Are student savings accounts exempt from maintenance fees?",
    "What is the penalty for a returned cheque due to insufficient funds?",
    "What is the maximum DBR for salaried customers?",
    "How can the credit card annual fee be waived?",
    "What is the KYC refresh frequency for high-risk customers?",
    "What is the daily transfer limit for new beneficiaries in the first 24 hours?",
    "A salaried customer has DBR 35%. Can they get a personal loan? Why?"
]

if __name__ == "__main__":
    for q in questions:
        print("\n" + "=" * 90)
        print("Q:", q)
        ans = rag_chain.invoke(q)
        print("A:", ans)


Q: What is the minimum monthly average balance for a savings account?
A: The minimum monthly average balance for a savings account is SAR 3,000.

Q: Are student savings accounts exempt from maintenance fees?
A: Yes, student savings accounts are exempt from maintenance fees.

Q: What is the penalty for a returned cheque due to insufficient funds?
A: A penalty of SAR 150 is applied per incident for a returned cheque due to insufficient funds.

Q: What is the maximum DBR for salaried customers?
A: The maximum Debt Burden Ratio (DBR) for salaried customers is 33%.

Q: How can the credit card annual fee be waived?
A: The credit card annual fee can be waived if the yearly spend exceeds SAR 20,000.

Q: What is the KYC refresh frequency for high-risk customers?
A: The KYC refresh frequency for high-risk customers is every 12 months.

Q: What is the daily transfer limit for new beneficiaries in the first 24 hours?
A: The daily transfer limit for new beneficiaries in the first 24 hours is SAR 5

# 🎓 Assignment: Build a Domain-Specific RAG AI Assistant

## Objective
Build a **Retrieval-Augmented Generation (RAG) system** that answers questions using **your domain data**.

You must use:
- LangChain
- Vector database
- Your own knowledge document

Do NOT use generic examples.  
Solve a real problem from your domain / workplace / target industry.
